In [2]:
import pandas as pd
import numpy as np
import ruptures as rpt


# ==========================================
# 1. Load TDA Dataset
# ==========================================

df = pd.read_csv(
    "data/market_data_tda.csv",
    parse_dates=["Date"],
    index_col="Date"
)

print("Dataset Shape:", df.shape)


# ==========================================
# 2. Configuration
# ==========================================

# Use only past 252 trading days (~1 year)
LOOKBACK = 252

# Minimum segment size
MIN_SIZE = 20

# PELT penalty
PENALTY = 10


# Signals representing market structure
signal_columns = [

    "SP500_Return",

    "SP500_Volatility_20",

    "VIX",

    "TDA_H1_TotalPersistence",

    "TDA_H1_Entropy"
]


# ==========================================
# 3. Output Arrays
# ==========================================

n = len(df)

pelt_change = np.zeros(n)

days_since_change = np.full(
    n,
    np.nan
)

change_strength = np.zeros(n)


print(
    "\nRunning Causal Rolling PELT..."
)


# ==========================================
# 4. Rolling PELT
# ==========================================

for i in range(
    LOOKBACK - 1,
    n
):

    start = (
        i - LOOKBACK + 1
    )

    window = df.iloc[
        start:i + 1
    ][signal_columns].copy()


    # --------------------------------------
    # Standardize inside historical window
    # --------------------------------------

    mean = window.mean()

    std = (
        window.std()
        .replace(0, 1)
    )

    scaled = (
        (window - mean) / std
    )

    signal = scaled.values


    # --------------------------------------
    # Run PELT
    # --------------------------------------

    algo = rpt.Pelt(
        model="rbf",
        min_size=MIN_SIZE,
        jump=5
    )

    breakpoints = algo.fit(
        signal
    ).predict(
        pen=PENALTY
    )


    # Last breakpoint == end of window
    real_breakpoints = [
        bp for bp in breakpoints
        if bp < len(window)
    ]


    # --------------------------------------
    # Extract Most Recent Change Point
    # --------------------------------------

    if len(real_breakpoints) > 0:

        last_change = (
            real_breakpoints[-1]
        )

        distance = (
            len(window) - last_change
        )

        days_since_change[i] = distance


        # Recent structural change indicator
        if distance <= 10:

            pelt_change[i] = 1


        # ----------------------------------
        # Change Strength
        # ----------------------------------

        before_start = max(
            0,
            last_change - 20
        )

        before = signal[
            before_start:last_change
        ]

        after = signal[
            last_change:
        ]


        if (
            len(before) > 0
            and len(after) > 0
        ):

            before_mean = (
                before.mean(axis=0)
            )

            after_mean = (
                after.mean(axis=0)
            )

            change_strength[i] = (
                np.linalg.norm(
                    after_mean -
                    before_mean
                )
            )


    # --------------------------------------
    # Progress
    # --------------------------------------

    processed = (
        i - LOOKBACK + 2
    )

    if processed % 500 == 0:

        print(
            f"Processed "
            f"{processed}/"
            f"{n - LOOKBACK + 1}"
        )


# ==========================================
# 5. Add PELT Features
# ==========================================

df[
    "PELT_Recent_Change"
] = pelt_change

df[
    "PELT_Days_Since_Change"
] = days_since_change

df[
    "PELT_Change_Strength"
] = change_strength


# ==========================================
# 6. Handle Initial / No Change Values
# ==========================================

# Large value means no recent detected change

df[
    "PELT_Days_Since_Change"
] = df[
    "PELT_Days_Since_Change"
].fillna(
    LOOKBACK
)


# ==========================================
# 7. Validation
# ==========================================

print(
    "\nPELT Recent Change Distribution:"
)

print(
    df[
        "PELT_Recent_Change"
    ].value_counts()
)


print(
    "\nPELT Feature Statistics:"
)

print(
    df[
        [
            "PELT_Days_Since_Change",
            "PELT_Change_Strength"
        ]
    ].describe()
)


# ==========================================
# 8. Crash vs Normal Comparison
# ==========================================

print(
    "\nAverage PELT Features by Crash Risk:"
)

print(
    df.groupby(
        "Crash_Risk"
    )[
        [
            "PELT_Recent_Change",
            "PELT_Days_Since_Change",
            "PELT_Change_Strength"
        ]
    ].mean()
)


# ==========================================
# 9. Known Crisis Period Check
# ==========================================

for name, period in {

    "Financial Crisis":
    ("2008-08-01", "2009-03-31"),

    "COVID Crash":
    ("2020-01-01", "2020-05-31")

}.items():

    start, end = period

    subset = df.loc[
        start:end
    ]

    print(
        f"\n{name}:"
    )

    print(
        subset[
            [
                "SP500",
                "VIX",
                "Crash_Risk",
                "PELT_Recent_Change",
                "PELT_Days_Since_Change",
                "PELT_Change_Strength"
            ]
        ]
        .sort_values(
            "PELT_Change_Strength",
            ascending=False
        )
        .head(10)
    )


# ==========================================
# 10. Save Dataset
# ==========================================

df.to_csv(
    "data/market_data_tda_pelt.csv"
)

print(
    "\nFinal Shape:",
    df.shape
)

print(
    "\nSaved:"
    " data/market_data_tda_pelt.csv"
)

Dataset Shape: (6423, 38)

Running Causal Rolling PELT...
Processed 500/6172
Processed 1000/6172
Processed 1500/6172
Processed 2000/6172
Processed 2500/6172
Processed 3000/6172
Processed 3500/6172
Processed 4000/6172
Processed 4500/6172
Processed 5000/6172
Processed 5500/6172
Processed 6000/6172

PELT Recent Change Distribution:
PELT_Recent_Change
0.0    6423
Name: count, dtype: int64

PELT Feature Statistics:
       PELT_Days_Since_Change  PELT_Change_Strength
count             6423.000000           6423.000000
mean               139.006850              1.919604
std                 76.548462              1.157249
min                 22.000000              0.000000
25%                 72.000000              1.554614
50%                122.000000              2.147445
75%                212.000000              2.740185
max                252.000000              4.802750

Average PELT Features by Crash Risk:
            PELT_Recent_Change  PELT_Days_Since_Change  PELT_Change_Strength
Cra

In [3]:
import pandas as pd
import numpy as np

# Load existing PELT output
df = pd.read_csv(
    "data/market_data_tda_pelt.csv",
    parse_dates=["Date"],
    index_col="Date"
)

print("Shape:", df.shape)


# ==========================================
# 1. Inspect Days Since Change
# ==========================================

print("\nDays Since Change Statistics:")

print(
    df["PELT_Days_Since_Change"]
    .describe(
        percentiles=[
            .10, .25, .50, .75, .90, .95
        ]
    )
)


# ==========================================
# 2. Create Better Recency Features
# ==========================================

# Change detected within past month
df["PELT_Change_30D"] = (
    df["PELT_Days_Since_Change"] <= 30
).astype(int)

# Change detected within past 60 trading days
df["PELT_Change_60D"] = (
    df["PELT_Days_Since_Change"] <= 60
).astype(int)


# ==========================================
# 3. Continuous Recency Score
# ==========================================

# Recent change -> score near 1
# Old change -> score approaches 0

df["PELT_Recency_Score"] = np.exp(
    -df["PELT_Days_Since_Change"] / 30
)


# ==========================================
# 4. Combined Structural Stress
# ==========================================

df["PELT_Structural_Stress"] = (
    df["PELT_Change_Strength"]
    *
    df["PELT_Recency_Score"]
)


# ==========================================
# 5. Validation
# ==========================================

print("\n30-Day Change Distribution:")

print(
    df["PELT_Change_30D"]
    .value_counts()
)


print("\n60-Day Change Distribution:")

print(
    df["PELT_Change_60D"]
    .value_counts()
)


print("\nCrash vs Normal:")

print(
    df.groupby("Crash_Risk")[
        [
            "PELT_Days_Since_Change",
            "PELT_Change_Strength",
            "PELT_Recency_Score",
            "PELT_Structural_Stress"
        ]
    ].mean()
)


# ==========================================
# 6. Save
# ==========================================

df.to_csv(
    "data/market_data_tda_pelt_fixed.csv"
)

print(
    "\nFinal Shape:",
    df.shape
)

print(
    "\nSaved:"
    " data/market_data_tda_pelt_fixed.csv"
)

Shape: (6423, 41)

Days Since Change Statistics:
count    6423.000000
mean      139.006850
std        76.548462
min        22.000000
10%        42.000000
25%        72.000000
50%       122.000000
75%       212.000000
90%       252.000000
95%       252.000000
max       252.000000
Name: PELT_Days_Since_Change, dtype: float64

30-Day Change Distribution:
PELT_Change_30D
0    6202
1     221
Name: count, dtype: int64

60-Day Change Distribution:
PELT_Change_60D
0    5291
1    1132
Name: count, dtype: int64

Crash vs Normal:
            PELT_Days_Since_Change  PELT_Change_Strength  PELT_Recency_Score  \
Crash_Risk                                                                     
0                       138.470779              1.931799            0.068133   
1                       151.562738              1.633989            0.081800   

            PELT_Structural_Stress  
Crash_Risk                          
0                         0.184446  
1                         0.252607  

Final